# HackSLU 2026 — Pothole Model Fine-Tuning

**Before running:**
1. Go to **Runtime → Change runtime type → GPU → A100** (or V100)
2. Add your Roboflow API key as a Colab secret named `ROBOFLOW_API_KEY`:
   - Click the 🔑 icon in the left sidebar → Add new secret
3. Run all cells in order

In [ ]:
# Cell 1 — Check GPU
!nvidia-smi

In [ ]:
# Cell 2 — Install dependencies
!pip install -q ultralytics roboflow python-dotenv

In [ ]:
# Cell 3 — Config (edit these if needed)
ROBOFLOW_WORKSPACE = "major-project-ye1u4"
ROBOFLOW_PROJECT   = "pavement-distress-analysis-siokj"
ROBOFLOW_VERSION   = 1

BASE_MODEL  = "yolov8m.pt"
EPOCHS      = 20
IMAGE_SIZE  = 640
BATCH_SIZE  = 32   # increase to 64 on A100
RUN_NAME    = "pothole_finetune"

In [ ]:
# Cell 4 — Load API key from Colab secrets
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
print("API key loaded:", "YES" if ROBOFLOW_API_KEY else "NO — add it in the 🔑 secrets panel")

In [ ]:
# Cell 5 — Download dataset from Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov8")
print("Dataset ready at:", dataset.location)

In [ ]:
# Cell 6 — Train
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    name=RUN_NAME,
    exist_ok=True,
)
print("Training complete!")

In [ ]:
# Cell 7 — Download best.pt to your computer
from google.colab import files
files.download(f"runs/detect/{RUN_NAME}/weights/best.pt")